In [0]:
df_bronze = spark.read.table("nyc.default.bronze_yellow_taxi")
print(f"Row Count: {df_bronze.count():,}")

In [0]:
display(df_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
df_silver = (df_bronze.withColumn("vendor_id", F.col("VendorID").cast(IntegerType()))\
    .withColumn("pickup_datetime",          F.col("tpep_pickup_datetime").cast(TimestampType()))\
    .withColumn("dropoff_datetime",         F.col("tpep_dropoff_datetime").cast(TimestampType()))\
    .withColumn("passenger_count",          F.col("passenger_count").cast(IntegerType()))\
    .withColumn("trip_distance",            F.col("trip_distance").cast(DoubleType()))\
    .withColumn("ratecode_id",              F.col("RatecodeID").cast(IntegerType()))\
    .withColumn("store_and_fwd_flag",       F.col("store_and_fwd_flag").cast("string"))\
    .withColumn("pu_location_id",           F.col("PULocationID").cast(IntegerType()))\
    .withColumn("do_location_id",           F.col("DOLocationID").cast(IntegerType()))\
    .withColumn("payment_type",             F.col("payment_type").cast(IntegerType()))\
    .withColumn("fare_amount",              F.col("fare_amount").cast(DoubleType()))\
    .withColumn("extra",                    F.col("extra").cast(DoubleType()))\
    .withColumn("mta_tax",                  F.col("mta_tax").cast(DoubleType()))\
    .withColumn("tip_amount",               F.col("tip_amount").cast(DoubleType()))\
    .withColumn("tolls_amount",             F.col("tolls_amount").cast(DoubleType()))\
    .withColumn("improvement_surcharge",    F.col("improvement_surcharge").cast(DoubleType()))\
    .withColumn("total_amount",             F.col("total_amount").cast(DoubleType()))\
    .withColumn("congestion_surcharge",     F.col("congestion_surcharge").cast(DoubleType()))\
    .withColumn("airport_fee",              F.col("Airport_fee").cast(DoubleType()))\
    .withColumn("cbd_congestion_fee",       F.col("cbd_congestion_fee").cast(DoubleType()))\
    .drop(
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "RatecodeID", "PULocationID", "DOLocationID",
        "Airport_fee"
    )
)

In [0]:
print(f"After cast row count: {df_silver.count():,}")

In [0]:
# Cmd 3 — Derive trip_duration_minutes
df_derived = df_silver.withColumn(
    "trip_duration_minutes",
    F.round(
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60,
        2
    )
)

In [0]:
# Cmd 4 — Filter bad rows
# Rules:
# - pickup before dropoff (no negative or zero duration)
# - trip_duration between 1 and 180 minutes (< 1 min = meter error; > 3hrs = outlier)
# - trip_distance > 0
# - fare_amount > 0
# - passenger_count between 1 and 6
# - pickup in January 2026 only (matches our source file)

df_silver_final = (
    df_derived
    .filter(F.col("pickup_datetime") < F.col("dropoff_datetime"))
    .filter(F.col("trip_duration_minutes").between(1, 180))
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(
        (F.col("pickup_datetime") >= "2026-01-01") &
        (F.col("pickup_datetime") < "2026-02-01")
    )
)

silver_count = df_silver_final.count()
print(f"Silver row count after filtering: {silver_count:,}")
print(f"Rows dropped: {df_bronze.count() - silver_count:,}")

In [0]:
# Cmd 5 — Write to Silver Delta table
(
    df_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc.default.silver_yellow_taxi")
)

print("silver_yellow_taxi written successfully.")

In [0]:
# Cmd 6 — Verify
df_check = spark.read.format("delta").table("nyc.default.silver_yellow_taxi")
print(f"Final silver row count: {df_check.count():,}")
df_check.printSchema()
display(df_check.limit(10))